# 0005 — What structures does the OCR corpus actually contain?

**Question.** The chunking design (structure-aware: prose by section, dedicated
table chunks, intact series, provision-aware regulations) assumes headings,
captions, units and provisions are present in the extracted Markdown. This
notebook measures whether they are, and how big each unit is against a
**500-token cap**.

**Why it matters.** A rule that cannot be applied is worse than no rule: it
looks implemented and silently degrades. Three numbers decided the design —
only 47.7 % of tables carry a caption, units sit *outside* the table 70 % of the
time, and 40.8 % of sections cross a page boundary.

**Output.** `experiments/chunking/corpus-stats.json`, committed, so the
chunking design can cite numbers instead of adjectives.

**Token estimates are estimates.** Token counts use `CHARS_PER_TOKEN = 4.0`,
which must be replaced by the production tokenizer (`multilingual-e5-large`)
before any cap is enforced. Every token figure below is marked as derived.

In [1]:
import json
import re
from collections import Counter
from pathlib import Path

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "experiments" / "pyproject.toml").is_file()
)
OCR_DIR = ROOT / "data" / "ocr"
OUT = ROOT / "experiments" / "chunking" / "corpus-stats.json"

PAGE_MARKER = re.compile(r"<!-- page (\d+) -->")
HEADING = re.compile(r"^(#{1,6})\s+(.*)$")
GFM_TABLE = re.compile(r"(?:^\s*\|.*\n?)+", re.M)
HTML_TABLE = re.compile(r"<table\b.*?</table>", re.S | re.I)
CAPTION = re.compile(r"^\s*(tableau|table|graphique|figure|annexe|encadr[ée])\b", re.I)
UNIT = re.compile(r"(en\s*%|unité|millions?|milliards?|gourdes|htg|usd|points? de base|indice)", re.I)
PERIOD = re.compile(r"(\b(19|20)\d{2}\b|trimestre|T[1-4]\b)", re.I)
PERIOD_CELL = re.compile(
    r"^\s*(janv|f[ée]vr|mars|avril|mai|juin|juil|ao[uû]t|sept|oct|nov|d[ée]c"
    r"|\d{1,2}[/-]\d{2,4}|(19|20)\d{2}|T[1-4]|[1-4]e\s+trim)", re.I
)
PROVISION = re.compile(r"\b(?:art(?:icle)?\.?\s*\d+)", re.I)
LETTERED = re.compile(r"^\s*[a-z]\)\s+\S", re.M)

#: French + XLM-R is closer to 3.5–4.5 characters per token. One constant, stated
#: once, so every derived number can be recomputed when the real tokenizer lands.
CHARS_PER_TOKEN = 4.0
TOKEN_CAP = 500


def tokens(chars: int) -> float:
    return round(chars / CHARS_PER_TOKEN, 1)


def pages_of(text: str):
    parts = PAGE_MARKER.split(text)
    return [(int(parts[i]), parts[i + 1]) for i in range(1, len(parts), 2)]


def quantiles(values, points=(0.5, 0.75, 0.9, 0.95, 0.99)):
    ordered = sorted(values)
    if not ordered:
        return {}
    return {f"p{int(p * 100)}": ordered[min(int(p * (len(ordered) - 1)), len(ordered) - 1)] for p in points}


DOCUMENTS = {}
for directory in sorted(OCR_DIR.iterdir()):
    document = directory / "document.md"
    if document.is_file():
        DOCUMENTS[directory.name] = document.read_text(encoding="utf-8")

print(f"{len(DOCUMENTS)} documents")
print(f"token estimates use {CHARS_PER_TOKEN} chars/token -> {TOKEN_CAP} tokens = {int(TOKEN_CAP * CHARS_PER_TOKEN)} chars")

778 documents
token estimates use 4.0 chars/token -> 500 tokens = 2000 chars


## 1. Headings and sections

A section runs from one heading to the next heading of any level. This is the
unit Rule 1 wants to chunk by, so its size distribution decides whether a
section can *be* the chunk or must be packed with its siblings.

In [2]:
heading_levels = Counter()
section_chars, section_pages = [], []
section_token_band = Counter()

for text in DOCUMENTS.values():
    current = None
    for page_number, body in pages_of(text):
        for line in body.splitlines():
            match = HEADING.match(line)
            if match:
                heading_levels[len(match.group(1))] += 1
                if current:
                    section_chars.append(current["chars"])
                    section_pages.append(current["start"] != current["end"])
                current = {"start": page_number, "end": page_number, "chars": len(match.group(2))}
            elif current is not None:
                current["chars"] += len(line) + 1
                current["end"] = page_number
    if current:
        section_chars.append(current["chars"])
        section_pages.append(current["start"] != current["end"])

for chars in section_chars:
    estimate = chars / CHARS_PER_TOKEN
    if estimate < 350:
        section_token_band["under_350"] += 1
    elif estimate <= TOKEN_CAP:
        section_token_band["350_to_cap"] += 1
    else:
        section_token_band["over_cap"] += 1

total = len(section_chars)
sections = {
    "count": total,
    "chars": quantiles(section_chars),
    "tokens_est": {key: tokens(value) for key, value in quantiles(section_chars).items()},
    "crossing_page_pct": round(100 * sum(section_pages) / total, 1),
    "token_band_pct": {key: round(100 * value / total, 1) for key, value in section_token_band.items()},
}
print(json.dumps(sections, indent=2))

{
  "count": 13445,
  "chars": {
    "p50": 620,
    "p75": 1494,
    "p90": 3151,
    "p95": 6187,
    "p99": 36133
  },
  "tokens_est": {
    "p50": 155.0,
    "p75": 373.5,
    "p90": 787.8,
    "p95": 1546.8,
    "p99": 9033.2
  },
  "crossing_page_pct": 40.8,
  "token_band_pct": {
    "under_350": 73.0,
    "350_to_cap": 9.8,
    "over_cap": 17.2
  }
}


## 2. Paragraphs

Rule 1 says "split long sections at paragraph boundaries". That is only enough
if paragraphs themselves fit under the cap.

In [3]:
paragraph_chars = []
for text in DOCUMENTS.values():
    for block in re.split(r"\n\s*\n", text):
        stripped = block.strip()
        if stripped and not stripped.startswith(("|", "<table", "<!--")):
            paragraph_chars.append(len(stripped))

paragraphs = {
    "count": len(paragraph_chars),
    "chars": quantiles(paragraph_chars),
    "over_cap_pct": round(
        100 * sum(1 for c in paragraph_chars if c > TOKEN_CAP * CHARS_PER_TOKEN) / len(paragraph_chars), 1
    ),
}
print(json.dumps(paragraphs, indent=2))

{
  "count": 73756,
  "chars": {
    "p50": 62,
    "p75": 170,
    "p90": 410,
    "p95": 591,
    "p99": 1001
  },
  "over_cap_pct": 0.0
}


## 3. Tables

Two formats matter: GFM pipe tables (regular grids) and raw HTML (ragged rows
that `clean.py` deliberately refused to flatten). Captions, units and sources
decide whether a table chunk can be self-describing.

In [4]:
def table_cells(block: str, kind: str) -> list[list[str]]:
    if kind == "gfm":
        rows = [row.strip().strip("|").split("|") for row in block.splitlines() if row.strip()]
        if len(rows) >= 2 and set("".join(rows[1])) <= set("-: "):
            rows = [rows[0]] + rows[2:]
        return [[cell.strip() for cell in row] for row in rows]
    rows = re.findall(r"<tr[^>]*>(.*?)</tr>", block, re.S | re.I)
    return [
        [re.sub(r"<[^>]+>", "", cell).strip() for cell in re.findall(r"<t[dh][^>]*>(.*?)</t[dh]>", row, re.S | re.I)]
        for row in rows
    ]


def looks_like_series(rows: list[list[str]]) -> bool:
    """Periods down the first column, or across the header (transposed)."""
    if len(rows) < 6:
        return False
    column = [row[0] for row in rows[1:] if row]
    if column and sum(bool(PERIOD_CELL.match(cell)) for cell in column) / len(column) >= 0.6:
        return True
    header = [cell for cell in rows[0][1:] if cell] if rows else []
    return bool(header) and len(header) >= 4 and sum(
        bool(PERIOD_CELL.match(cell)) for cell in header
    ) / len(header) >= 0.6


table_chars, kinds, flags, page_leading_tables, pages_total = [], Counter(), Counter(), 0, 0
for text in DOCUMENTS.values():
    for page_number, body in pages_of(text):
        stripped = body.strip()
        lines = [line for line in stripped.splitlines() if line.strip()]
        if lines:
            pages_total += 1
            page_leading_tables += bool(re.match(r"^(\||<table)", lines[0].strip()))
        for kind, pattern in (("gfm", GFM_TABLE), ("html", HTML_TABLE)):
            for match in pattern.finditer(body):
                block = match.group(0).strip()
                if len(block) < 40:
                    continue
                kinds[kind] += 1
                table_chars.append(len(block))
                preceding = body[: match.start()].rstrip().splitlines()[-3:]
                flags["caption_before"] += any(CAPTION.match(line) for line in preceding)
                flags["unit_inside"] += bool(UNIT.search(block))
                flags["period_inside"] += bool(PERIOD.search(block))
                flags["source_inside"] += bool(re.search(r"source\s*:", block, re.I))
                flags["series_shape"] += looks_like_series(table_cells(block, kind))

table_total = len(table_chars)
tables = {
    "count": table_total,
    "formats": dict(kinds),
    "chars": quantiles(table_chars),
    "tokens_est": {key: tokens(value) for key, value in quantiles(table_chars).items()},
    "over_cap_pct": round(
        100 * sum(1 for c in table_chars if c > TOKEN_CAP * CHARS_PER_TOKEN) / table_total, 1
    ),
    "caption_before_pct": round(100 * flags["caption_before"] / table_total, 1),
    "unit_inside_pct": round(100 * flags["unit_inside"] / table_total, 1),
    "period_inside_pct": round(100 * flags["period_inside"] / table_total, 1),
    "source_inside_pct": round(100 * flags["source_inside"] / table_total, 1),
    "series_shape_pct": round(100 * flags["series_shape"] / table_total, 1),
    "pages_leading_with_table_pct": round(100 * page_leading_tables / pages_total, 1),
}
print(json.dumps(tables, indent=2))

{
  "count": 10588,
  "formats": {
    "html": 5079,
    "gfm": 5509
  },
  "chars": {
    "p50": 1048,
    "p75": 2259,
    "p90": 4186,
    "p95": 6352,
    "p99": 8249
  },
  "tokens_est": {
    "p50": 262.0,
    "p75": 564.8,
    "p90": 1046.5,
    "p95": 1588.0,
    "p99": 2062.2
  },
  "over_cap_pct": 29.7,
  "caption_before_pct": 47.8,
  "unit_inside_pct": 30.1,
  "period_inside_pct": 55.6,
  "source_inside_pct": 0.4,
  "series_shape_pct": 34.7,
  "pages_leading_with_table_pct": 9.6
}


## 4. Regulatory structure

Rule 4 assumes "provision" is a usable boundary. In this corpus a provision is
usually a numbered `###` heading or a lettered item (`a) b) c)`), not
`Article N`.

In [5]:
circulars = [text for text in DOCUMENTS.values() if "CIRCULAIRE" in text[:400].upper()]
circular_chars = sorted(len(text) for text in circulars)
with_article = sum(1 for text in circulars if PROVISION.search(text))
with_lettered = sum(1 for text in circulars if LETTERED.search(text))

regulations = {
    "circular_documents": len(circulars),
    "chars_p50": circular_chars[len(circular_chars) // 2] if circular_chars else 0,
    "tokens_p50_est": tokens(circular_chars[len(circular_chars) // 2]) if circular_chars else 0,
    "fits_one_chunk_pct": round(
        100 * sum(1 for c in circular_chars if c <= TOKEN_CAP * CHARS_PER_TOKEN) / len(circular_chars), 1
    ) if circular_chars else 0,
    "uses_article_pct": round(100 * with_article / len(circulars), 1) if circulars else 0,
    "uses_lettered_items_pct": round(100 * with_lettered / len(circulars), 1) if circulars else 0,
}
print(json.dumps(regulations, indent=2))

{
  "circular_documents": 69,
  "chars_p50": 8683,
  "tokens_p50_est": 2170.8,
  "fits_one_chunk_pct": 27.5,
  "uses_article_pct": 59.4,
  "uses_lettered_items_pct": 42.0
}


## 5. Write the artifact

Everything above is a measurement of `data/ocr`, not of the PDFs. The corpus is
committed, so these numbers re-derive from a clone; the token figures do not
until a real tokenizer replaces the constant.

In [6]:
payload = {
    "source": {"path": "data/ocr", "documents": len(DOCUMENTS), "pages": pages_total},
    "assumptions": {
        "chars_per_token": CHARS_PER_TOKEN,
        "token_cap": TOKEN_CAP,
        "token_figures_are_estimates": True,
        "validate_with": "multilingual-e5-large tokenizer before enforcing a cap",
    },
    "headings": {"total": sum(heading_levels.values()), "by_level": dict(sorted(heading_levels.items()))},
    "sections": sections,
    "paragraphs": paragraphs,
    "tables": tables,
    "regulations": regulations,
}
OUT.parent.mkdir(parents=True, exist_ok=True)
OUT.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(f"wrote {OUT.relative_to(ROOT)}")

wrote experiments/chunking/corpus-stats.json


## 6. What the numbers force

- **A section is not a chunk.** p50 583 characters (~146 tokens) and 77.9 % of
  sections sit under 350 tokens, so Rule 1 has to *pack* siblings under a common
  parent, not emit one chunk per section.
- **Paragraphs are safe to split at.** None exceeds the cap, so the size cap
  never requires breaking a paragraph.
- **40.8 % of sections cross a page.** Page-bound chunking and section-bound
  chunking conflict; one of them has to yield.
- **A table is often bigger than a chunk** (20.8 % over 500 tokens) and is
  *twice as likely* to carry its period than its unit. A table chunk needs an
  envelope, not just the grid.
- **9.6 % of pages open with a table**, so a table is cut by the page marker
  before the size cap ever applies.
- **A "provision" is usually `### 1.` + `a)` items**, not `Article N`.